# Step 3: The XY Model and the Kosterlitz-Thouless Transition

In notebooks 01 and 02 we studied a model with a **discrete** $\mathbb{Z}_2$ symmetry. Now each spin becomes a continuous angle $\theta_i \in [0, 2\pi)$, promoting the symmetry to $U(1)$ — continuous rotations in the plane. This single change leads to qualitatively new physics: no conventional ordered phase in 2D, but a topological phase transition driven by vortex unbinding.

**What you will do:**
- Build the XY model and implement Metropolis for continuous spin updates
- Visualise the angle field and observe the contrast between quasi-ordered and disordered configurations
- Implement the vortex-detection algorithm and map defect positions
- Measure the **vortex density** across a temperature sweep and watch it rise sharply near $T_{BKT}$
- Compute the **helicity modulus** $\Upsilon$ and identify the BKT transition from its universal jump $\Upsilon(T_{BKT}^-) = 2T_{BKT}/\pi$

**Prerequisites:** Notebooks 01 and 02.

## 3.1  The XY Model

The Hamiltonian is:

$$H = -J \sum_{\langle i,j \rangle} \cos(\theta_i - \theta_j)$$

Each spin is an angle $\theta_i \in [0, 2\pi)$ representing the direction of a unit vector $\vec{s}_i = (\cos\theta_i, \sin\theta_i)$ in the plane. The energy is minimised when all angles are equal and increases as $J(1 - \cos\Delta\theta) \approx J(\Delta\theta)^2/2$ for small misalignments — the XY model behaves locally like an elastic medium for the angle field.

The symmetry is $U(1)$: rotating all spins by the same angle $\phi$ leaves $H$ unchanged. The **Mermin-Wagner theorem** tells us this continuous symmetry cannot be spontaneously broken at any finite $T$ in 2D — there is no ferromagnetic phase. But, as Kosterlitz and Thouless showed, there is something more subtle.

We work with $J = 1$ and $k_B = 1$ throughout.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

J = 1.0
T_BKT = 0.893   # J = 1; see lecture notes §4

def spin_init(L):
    """Random angle configuration: theta_i in [0, 2*pi)."""
    return np.random.uniform(0, 2 * np.pi, size=(L, L))

def total_energy_xy(angles, L, J=1.0):
    """Total energy E = -J * sum_{<ij>} cos(theta_i - theta_j)."""
    right = np.roll(angles, -1, axis=1)
    down  = np.roll(angles, -1, axis=0)
    return -J * np.sum(np.cos(angles - right) + np.cos(angles - down))

# Sanity check: fully aligned state should have E = -2*J*N
L_test   = 8
aligned  = np.zeros((L_test, L_test))
E_check  = total_energy_xy(aligned, L_test)
E_expect = -2 * J * L_test**2
print(f"Aligned state energy: {E_check:.1f}  (expected {E_expect:.1f})")

## 3.2  Metropolis for Continuous Spins

The proposal is an angular perturbation at a randomly chosen site:

$$\theta_i \to \theta_i + \delta, \qquad \delta \sim \mathrm{Uniform}(-\delta_{\max}, \delta_{\max})$$

The energy change involves only the four neighbours of site $i$:

$$\Delta E = -J \sum_{n \in \mathrm{nn}(i)} \bigl[\cos(\theta_i + \delta - \theta_n) - \cos(\theta_i - \theta_n)\bigr]$$

Accept if $\Delta E \leq 0$; otherwise accept with probability $e^{-\beta \Delta E}$. The rule is identical to the Ising case.

**Tuning $\delta_{\max}$:** Too small and the chain barely moves; too large and most proposals are rejected. A value near 1 radian gives reasonable acceptance across the temperatures we study.

For efficiency we use a **checkerboard decomposition**: the square lattice splits into two sublattices (even and odd sites by $(i+j) \bmod 2$) such that every even site's neighbours are all odd, and vice versa. Both sublattices can therefore be updated in a single vectorised pass.

In [ ]:
def metropolis_xy_sweep(angles, L, beta, J=1.0, delta_max=1.0):
    """
    One full lattice sweep using checkerboard decomposition.
    Both sublattices (even/odd by (i+j) % 2) are visited once per sweep.
    Returns the acceptance rate (useful for tuning delta_max).
    """
    i_idx, j_idx = np.meshgrid(np.arange(L), np.arange(L), indexing='ij')
    n_accept = 0

    for parity in (0, 1):
        mask = ((i_idx + j_idx) % 2 == parity)

        # Current interaction sum: cos(theta_i - theta_n) summed over 4 neighbours
        nn_cos_old = (np.cos(angles - np.roll(angles, -1, axis=1)) +
                      np.cos(angles - np.roll(angles,  1, axis=1)) +
                      np.cos(angles - np.roll(angles, -1, axis=0)) +
                      np.cos(angles - np.roll(angles,  1, axis=0)))

        # Proposed new angles
        delta      = np.random.uniform(-delta_max, delta_max, (L, L))
        new_angles = (angles + delta) % (2 * np.pi)

        # New interaction sum (neighbours unchanged in this half-sweep)
        nn_cos_new = (np.cos(new_angles - np.roll(angles, -1, axis=1)) +
                      np.cos(new_angles - np.roll(angles,  1, axis=1)) +
                      np.cos(new_angles - np.roll(angles, -1, axis=0)) +
                      np.cos(new_angles - np.roll(angles,  1, axis=0)))

        dE   = -J * (nn_cos_new - nn_cos_old)
        prob = np.where(dE <= 0, 1.0, np.exp(-beta * dE))
        accept = mask & (np.random.random((L, L)) < prob)
        angles[accept] = new_angles[accept]
        n_accept += np.sum(accept)

    return n_accept / (L * L)


def equilibrate(L, T, n_sweeps, J=1.0, delta_max=1.0):
    """Start from a random configuration and run n_sweeps Metropolis sweeps."""
    angles = spin_init(L)
    beta   = 1.0 / T
    for _ in range(n_sweeps):
        metropolis_xy_sweep(angles, L, beta, J, delta_max)
    return angles


# Check acceptance rates at two representative temperatures
for T_check in [0.5, 1.2]:
    a    = spin_init(32)
    rate = metropolis_xy_sweep(a, 32, 1.0 / T_check)
    print(f"T = {T_check:.1f}:  acceptance rate = {rate:.2f}")

## 3.3  Visualising the Spin Field

Below $T_{BKT} \approx 0.893$, spin-spin correlations decay algebraically — quasi-long-range order. Large regions share nearly the same angle and the colour field is smoothly varying. Above $T_{BKT}$, free vortices disorder the system and the colour field becomes patchy.

The background colour encodes $\theta_i$ as a hue in $[0, 2\pi)$. Arrows show the spin direction $\vec{s}_i = (\cos\theta_i, \sin\theta_i)$.

In [ ]:
L       = 32
N_EQUIL = 3000

fig, axes = plt.subplots(1, 2, figsize=(13, 6))

for ax, T in zip(axes, [0.5, 1.3]):
    angles = equilibrate(L, T, N_EQUIL)

    # Angle field as colour background
    ax.imshow(angles % (2 * np.pi), cmap='hsv', vmin=0, vmax=2 * np.pi,
              origin='lower', extent=[0, L, 0, L], alpha=0.7)

    # Arrows every 2 sites for readability
    step = 2
    ii   = np.arange(step // 2, L, step)
    jj   = np.arange(step // 2, L, step)
    Ji, Jj = np.meshgrid(jj, ii)
    U = np.cos(angles[ii[:, None], jj[None, :]])
    V = np.sin(angles[ii[:, None], jj[None, :]])
    ax.quiver(Ji, Jj, U, V, scale=24, headwidth=3, color='k', alpha=0.55)

    ax.set_xlim(0, L)
    ax.set_ylim(0, L)
    ax.set_aspect('equal')
    label = 'below' if T < T_BKT else 'above'
    ax.set_title(f"$T = {T}$  ({label} $T_{{BKT}}$)", fontsize=13)
    ax.axis('off')

plt.suptitle("XY spin field: angle (colour) and direction (arrows)", fontsize=14)
plt.tight_layout()
plt.show()

## 3.4  Detecting Vortices

A **vortex** is a configuration where the angle field winds by $\pm 2\pi$ around a closed loop. To find them, we check every elementary plaquette (four-site square) of the lattice:

1. Compute the angle difference along each of the four edges, wrapping each result into $(-\pi, \pi]$.
2. Sum the four wrapped differences. If the sum is near $+2\pi$ the plaquette contains a vortex; if near $-2\pi$, an antivortex; otherwise no defect.

The wrapping step is essential — it resolves the $2\pi$ periodicity of the angle and ensures the winding number is a topological invariant unchanged by smooth deformations of $\theta$.

**Below $T_{BKT}$:** vortices exist only in tightly bound pairs whose topological charges cancel — they are invisible to long-wavelength probes.  
**Above $T_{BKT}$:** pairs unbind and free vortices proliferate throughout the lattice.

In [ ]:
def find_vortices(angles, L):
    """
    Compute the topological charge of every elementary plaquette.

    Plaquette at (i, j) traverses corners:
        (i,j) -> (i,j+1) -> (i+1,j+1) -> (i+1,j) -> back

    Returns an integer array of shape (L, L) with values in {-1, 0, +1}.
    """
    def wrap(dtheta):
        """Constrain angle difference to (-pi, pi]."""
        return (dtheta + np.pi) % (2 * np.pi) - np.pi

    a_right = np.roll(angles, -1, axis=1)   # theta[i,   j+1]
    a_down  = np.roll(angles, -1, axis=0)   # theta[i+1, j  ]
    a_dr    = np.roll(a_right, -1, axis=0)  # theta[i+1, j+1]

    d1 = wrap(a_right - angles)    # right edge
    d2 = wrap(a_dr    - a_right)   # bottom-right edge
    d3 = wrap(a_down  - a_dr)      # bottom edge (traversed rightward)
    d4 = wrap(angles  - a_down)    # left edge (closing the loop)

    winding = (d1 + d2 + d3 + d4) / (2 * np.pi)
    return np.round(winding).astype(int)


# Visualise vortex positions at two temperatures
fig, axes = plt.subplots(1, 2, figsize=(13, 6))

for ax, T in zip(axes, [0.5, 1.3]):
    angles = equilibrate(L, T, N_EQUIL)
    charge = find_vortices(angles, L)

    ax.imshow(angles % (2 * np.pi), cmap='hsv', vmin=0, vmax=2 * np.pi,
              origin='lower', extent=[0, L, 0, L], alpha=0.65)

    vi, vj = np.where(charge ==  1)
    ai, aj = np.where(charge == -1)
    ax.scatter(vj + 0.5, vi + 0.5, c='white', s=90, marker='+',
               linewidths=2.5, zorder=3, label=f"+1 vortices: {len(vi)}")
    ax.scatter(aj + 0.5, ai + 0.5, c='black', s=90, marker='x',
               linewidths=2.5, zorder=3, label=f"\u22121 vortices: {len(ai)}")

    ax.set_xlim(0, L)
    ax.set_ylim(0, L)
    ax.set_aspect('equal')
    ax.set_title(f"$T = {T}$", fontsize=13)
    ax.legend(fontsize=9, loc='upper right')
    ax.axis('off')

plt.suptitle("Vortex (+, white) and antivortex (\u00d7, black) positions", fontsize=14)
plt.tight_layout()
plt.show()

## 3.5  Vortex Density across the Transition

Below $T_{BKT}$, vortices are bound into pairs and the free vortex density is exponentially small. Above $T_{BKT}$, pairs unbind and the density rises steeply. We measure the total vortex density:

$$n_v = \frac{1}{L^2} \sum_{\text{plaquettes}} |q|$$

and watch it activate near $T_{BKT}$.

Because the BKT correlation length diverges **exponentially** — not as a power law — finite-size effects are much more severe than for the Ising model. The rise in $n_v$ is broader and more rounded than the Ising magnetization curves.

In [ ]:
L_VORT      = 32
T_SWEEP     = np.linspace(0.40, 1.40, 16)
N_EQUIL_V   = 2000
N_MEAS_V    = 50    # snapshots per temperature
N_BETWEEN_V = 50    # sweeps between snapshots

vortex_density = []

print(f"{'T':>6}  |  vortex density")
print("-" * 26)
for T in T_SWEEP:
    angles = equilibrate(L_VORT, T, N_EQUIL_V)
    beta   = 1.0 / T

    nv_samples = []
    for _ in range(N_MEAS_V):
        for _ in range(N_BETWEEN_V):
            metropolis_xy_sweep(angles, L_VORT, beta)
        charge = find_vortices(angles, L_VORT)
        nv_samples.append(np.sum(np.abs(charge)) / L_VORT**2)

    rho = np.mean(nv_samples)
    vortex_density.append(rho)
    print(f"{T:6.2f}  |  {rho:.4f}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(T_SWEEP, vortex_density, 'o-', lw=1.5, ms=6, color='steelblue')
ax.axvline(T_BKT, color='k', ls='--', lw=1.2,
           label=f"$T_{{BKT}} \\approx {T_BKT}$")
ax.set_xlabel("Temperature $T$", fontsize=13)
ax.set_ylabel("Vortex density $n_v$", fontsize=13)
ax.set_title(f"Vortex proliferation at the BKT transition  ($L = {L_VORT}$)",
             fontsize=13)
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 3.6  The Helicity Modulus

The **helicity modulus** $\Upsilon$ measures the free-energy cost of twisting the angle field by a small angle $\phi$ across the system in the $x$-direction:

$$\Upsilon = \frac{1}{N} \frac{\partial^2 F}{\partial \phi^2}\Bigg|_{\phi = 0}$$

Evaluating this with linear response theory gives the expression we can measure from equilibrium snapshots:

$$N\,\Upsilon = J \left\langle \sum_{\langle ij \rangle_x} \cos(\theta_i - \theta_j) \right\rangle
- \frac{J^2}{T} \left\langle \left(\sum_{\langle ij \rangle_x} \sin(\theta_i - \theta_j)\right)^2 \right\rangle$$

where the sums run over **horizontal bonds** only.

- **Below $T_{BKT}$:** $\Upsilon > 0$ — the system resists the twist.
- **At $T_{BKT}$:** $\Upsilon$ drops discontinuously to zero.
- **Universal jump:** BKT theory gives the exact size of the discontinuity: $\Upsilon(T_{BKT}^-) = \frac{2}{\pi}T_{BKT}$.

On a finite lattice the jump is rounded, but curves for increasing $L$ all converge to intersect the line $\Upsilon = 2T/\pi$ near $T_{BKT}$.

In [ ]:
def helicity_sample(angles, L, J=1.0):
    """
    Compute the two fluctuation terms from one equilibrium snapshot.
    Accumulated over many snapshots to estimate Upsilon.
    """
    dtheta    = angles - np.roll(angles, -1, axis=1)   # horizontal bonds
    cos_term  = np.sum(np.cos(dtheta))
    sin_term2 = np.sum(np.sin(dtheta)) ** 2
    return cos_term, sin_term2


def sweep_helicity(L, T_values, n_equil, n_samples, n_between, J=1.0):
    """
    Temperature sweep of the helicity modulus for one lattice size.
    Returns array of Upsilon values, one per entry in T_values.
    """
    N       = L ** 2
    upsilon = np.zeros(len(T_values))

    for k, T in enumerate(T_values):
        angles = equilibrate(L, T, n_equil, J)
        beta   = 1.0 / T

        cos_acc, sin2_acc = [], []
        for _ in range(n_samples):
            for _ in range(n_between):
                metropolis_xy_sweep(angles, L, beta, J)
            ct, st2 = helicity_sample(angles, L, J)
            cos_acc.append(ct)
            sin2_acc.append(st2)

        upsilon[k] = (J / N) * (np.mean(cos_acc) - (J / T) * np.mean(sin2_acc))

    return upsilon


# ── Run the sweep ─────────────────────────────────────────────────────────────
L_VALUES    = [16, 24, 32]
T_HELIC     = np.linspace(0.40, 1.40, 20)
N_EQUIL_H   = 3000
N_SAMPLES_H = 200
N_BETWEEN_H = 20

results_helic = {}
for L in L_VALUES:
    print(f"L = {L} ...", end=' ', flush=True)
    results_helic[L] = sweep_helicity(
        L, T_HELIC, N_EQUIL_H, N_SAMPLES_H, N_BETWEEN_H
    )
    print("done")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

for L in L_VALUES:
    ax.plot(T_HELIC, results_helic[L], 'o-', ms=4, lw=1.5, label=f"$L = {L}$")

# Universal jump line: Upsilon = 2T/pi
T_line = np.linspace(0.3, 1.5, 300)
ax.plot(T_line, 2 * T_line / np.pi, 'k--', lw=1.5,
        label=r"$\Upsilon = 2T/\pi$  (universal jump)")

ax.axvline(T_BKT, color='grey', ls=':', lw=1.2,
           label=f"$T_{{BKT}} \\approx {T_BKT}$")

ax.set_xlabel("Temperature $T$", fontsize=13)
ax.set_ylabel(r"Helicity modulus $\Upsilon$", fontsize=13)
ax.set_title("Helicity modulus: the BKT transition and its universal jump",
             fontsize=13)
ax.set_ylim(bottom=-0.05)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Universal jump size at T_BKT:  Upsilon = 2*T_BKT/pi = {2*T_BKT/np.pi:.3f}")

## Exercises

1. **Acceptance rate tuning.** `metropolis_xy_sweep` returns the acceptance rate. Write a short loop that tries several values of `delta_max` at a given temperature and selects the one closest to 50%. How does the optimal `delta_max` depend on $T$?

2. **Correlation function.** Compute $G(r) = \langle \cos(\theta_0 - \theta_r) \rangle$ for separations $r = 0, 1, \ldots, L/2$ at $T = 0.6$ and $T = 1.1$. Below $T_{BKT}$, fit a power law $G(r) \sim r^{-\eta}$ and check whether $\eta < 1/4$. Above $T_{BKT}$, fit an exponential to extract the correlation length $\xi$.

3. **Vortex pairs.** At $T = 0.6$, pair each $+1$ vortex with its nearest $-1$ antivortex. Plot the histogram of pair separations. The lecture notes predict a logarithmic binding potential — does the distribution look consistent with that?

4. **System-size convergence.** Repeat the helicity modulus sweep for $L = 48$. As $L$ increases, the drop in $\Upsilon$ should sharpen and all curves should cross the line $\Upsilon = 2T/\pi$ at progressively closer temperatures. How precisely can you locate $T_{BKT}$ from your data?